# Credit Card Fraud Detection - Model Comparison

This notebook compares the performance of four machine learning models:
1. **Logistic Regression** (lr_model)
2. **Random Forest** (rf_model)
3. **XGBoost** (xgb_model)
4. **Artificial Neural Network** (ann_model)

All models are evaluated on the same test set using multiple metrics including ROC-AUC, AUPRC, Precision, Recall, and F1-Score.

### Step 1: Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    auc,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


### Step 2: Load All Saved Models

In [2]:
# Load all 4 models
lr_model = joblib.load('data/lr_model.pkl')
rf_model = joblib.load('data/rf_model.pkl')
xgb_model = joblib.load('data/xgb_model.pkl')
ann_model = joblib.load('data/ann_model.pkl')

print("✓ Logistic Regression model loaded")
print("✓ Random Forest model loaded")
print("✓ XGBoost model loaded")
print("✓ ANN model loaded")

FileNotFoundError: [Errno 2] No such file or directory: 'data/lr_model.pkl'

### Step 3: Load and Preprocess the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv("data/creditcard.csv")

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nClass distribution:")
print(df["Class"].value_counts())
print(f"\nFraud percentage: {df['Class'].mean() * 100:.4f}%")

In [ ]:
# Split features and target
X = df.drop("Class", axis=1)
y = df["Class"]

# Scale Amount and Time
scaler = StandardScaler()
X["Amount"] = scaler.fit_transform(X[["Amount"]])
X["Time"] = scaler.fit_transform(X[["Time"]])

print("✓ Data scaled")
print(f"  Amount - Mean: {X['Amount'].mean():.4f}, Std: {X['Amount'].std():.4f}")
print(f"  Time - Mean: {X['Time'].mean():.4f}, Std: {X['Time'].std():.4f}")

In [ ]:
# Train/test split (80/20) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTest set fraud ratio: {y_test.mean() * 100:.4f}%")

### Step 4: Generate Predictions and Scores for All Models

In [ ]:
# Generate predictions and probability scores for each model

# Logistic Regression
y_pred_lr = lr_model.predict(X_test)
y_scores_lr = lr_model.predict_proba(X_test)[:, 1]

# Random Forest
y_pred_rf = rf_model.predict(X_test)
y_scores_rf = rf_model.predict_proba(X_test)[:, 1]

# XGBoost
y_pred_xgb = xgb_model.predict(X_test)
y_scores_xgb = xgb_model.predict_proba(X_test)[:, 1]

# ANN
y_pred_ann = ann_model.predict(X_test)
y_scores_ann = ann_model.predict_proba(X_test)[:, 1]

print("✓ All predictions generated")

### Step 5: Plot ROC Curves for All Models

In [ ]:
# Calculate ROC curves for all models
# Create images directory if it doesn't exist
import os
os.makedirs('images', exist_ok=True)

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_scores_lr)
roc_auc_lr = roc_auc_score(y_test, y_scores_lr)

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_scores_rf)
roc_auc_rf = roc_auc_score(y_test, y_scores_rf)

fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_scores_xgb)
roc_auc_xgb = roc_auc_score(y_test, y_scores_xgb)

fpr_ann, tpr_ann, _ = roc_curve(y_test, y_scores_ann)
roc_auc_ann = roc_auc_score(y_test, y_scores_ann)

# Plot all ROC curves on one figure
plt.figure(figsize=(10, 8))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_lr:.4f})', linewidth=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_rf:.4f})', linewidth=2)
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {roc_auc_xgb:.4f})', linewidth=2)
plt.plot(fpr_ann, tpr_ann, label=f'ANN (AUC = {roc_auc_ann:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline', linewidth=1)

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('images/all_roc_curves.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nROC-AUC Scores:")
print(f"  Logistic Regression: {roc_auc_lr:.4f}")
print(f"  Random Forest:       {roc_auc_rf:.4f}")
print(f"  XGBoost:             {roc_auc_xgb:.4f}")
print(f"  ANN:                 {roc_auc_ann:.4f}")

### Step 6: Plot Precision-Recall Curves for All Models

In [ ]:
# Calculate Precision-Recall curves for all models
precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_scores_lr)
pr_auc_lr = auc(recall_lr, precision_lr)
avg_precision_lr = average_precision_score(y_test, y_scores_lr)

precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_scores_rf)
pr_auc_rf = auc(recall_rf, precision_rf)
avg_precision_rf = average_precision_score(y_test, y_scores_rf)

precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, y_scores_xgb)
pr_auc_xgb = auc(recall_xgb, precision_xgb)
avg_precision_xgb = average_precision_score(y_test, y_scores_xgb)

precision_ann, recall_ann, _ = precision_recall_curve(y_test, y_scores_ann)
pr_auc_ann = auc(recall_ann, precision_ann)
avg_precision_ann = average_precision_score(y_test, y_scores_ann)

# Plot all Precision-Recall curves on one figure
plt.figure(figsize=(10, 8))
plt.plot(recall_lr, precision_lr, label=f'Logistic Regression (AUPRC = {pr_auc_lr:.4f})', linewidth=2)
plt.plot(recall_rf, precision_rf, label=f'Random Forest (AUPRC = {pr_auc_rf:.4f})', linewidth=2)
plt.plot(recall_xgb, precision_xgb, label=f'XGBoost (AUPRC = {pr_auc_xgb:.4f})', linewidth=2)
plt.plot(recall_ann, precision_ann, label=f'ANN (AUPRC = {pr_auc_ann:.4f})', linewidth=2)

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('images/all_pr_curves.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nAUPRC Scores:")
print(f"  Logistic Regression: {pr_auc_lr:.4f}")
print(f"  Random Forest:       {pr_auc_rf:.4f}")
print(f"  XGBoost:             {pr_auc_xgb:.4f}")
print(f"  ANN:                 {pr_auc_ann:.4f}")

### Step 7: Comprehensive Model Comparison Table

In [ ]:
# Calculate all metrics for each model
metrics_data = {
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'ANN'],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_ann)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_ann)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_ann)
    ],
    'AUPRC': [
        pr_auc_lr,
        pr_auc_rf,
        pr_auc_xgb,
        pr_auc_ann
    ],
    'ROC-AUC': [
        roc_auc_lr,
        roc_auc_rf,
        roc_auc_xgb,
        roc_auc_ann
    ]
}

# Create comparison DataFrame
comparison_df = pd.DataFrame(metrics_data)

# Format the numbers to 4 decimal places
comparison_df_display = comparison_df.copy()
comparison_df_display['Precision'] = comparison_df_display['Precision'].apply(lambda x: f"{x:.4f}")
comparison_df_display['Recall'] = comparison_df_display['Recall'].apply(lambda x: f"{x:.4f}")
comparison_df_display['F1-Score'] = comparison_df_display['F1-Score'].apply(lambda x: f"{x:.4f}")
comparison_df_display['AUPRC'] = comparison_df_display['AUPRC'].apply(lambda x: f"{x:.4f}")
comparison_df_display['ROC-AUC'] = comparison_df_display['ROC-AUC'].apply(lambda x: f"{x:.4f}")

print("="*80)
print("MODEL PERFORMANCE COMPARISON")
print("="*80)
print(comparison_df_display.to_string(index=False))
print("="*80)

In [ ]:
# Create comparison bar chart
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Model Performance Comparison - All Metrics', fontsize=16, fontweight='bold')

metrics = ['Precision', 'Recall', 'F1-Score', 'AUPRC', 'ROC-AUC']
models = comparison_df['Model']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

# Plot each metric
for idx, metric in enumerate(metrics):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    values = comparison_df[metric]
    bars = ax.bar(range(len(models)), values, color=colors, alpha=0.7, edgecolor='black')
    
    # Highlight the best performer
    best_idx = values.idxmax()
    bars[best_idx].set_alpha(1.0)
    bars[best_idx].set_edgecolor('darkgreen')
    bars[best_idx].set_linewidth(3)
    
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(['LR', 'RF', 'XGB', 'ANN'], fontsize=10)
    ax.set_ylabel('Score', fontsize=10)
    ax.set_title(f'{metric}', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, v in enumerate(values):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

# Hide the extra subplot
axes[1, 2].axis('off')

# Add legend
legend_elements = [plt.Rectangle((0,0),1,1, fc=colors[i], alpha=0.7, edgecolor='black') 
                   for i in range(len(models))]
axes[1, 2].legend(legend_elements, models, loc='center', fontsize=11, title='Models', title_fontsize=12)

plt.tight_layout()
plt.savefig('images/comparison_bar_chart.png', bbox_inches='tight', dpi=150)
plt.show()

print("✓ Comparison bar chart saved to images/comparison_bar_chart.png")

### Step 8: Best Model Analysis

In [ ]:
# Find best model based on AUPRC and ROC-AUC
best_auprc_idx = comparison_df['AUPRC'].idxmax()
best_roc_idx = comparison_df['ROC-AUC'].idxmax()

best_auprc_model = comparison_df.loc[best_auprc_idx, 'Model']
best_auprc_score = comparison_df.loc[best_auprc_idx, 'AUPRC']

best_roc_model = comparison_df.loc[best_roc_idx, 'Model']
best_roc_score = comparison_df.loc[best_roc_idx, 'ROC-AUC']

print("\n" + "="*80)
print("BEST MODEL ANALYSIS")
print("="*80)

print(f"\n🏆 Best Model by AUPRC: {best_auprc_model}")
print(f"   AUPRC Score: {best_auprc_score:.4f}")

print(f"\n🏆 Best Model by ROC-AUC: {best_roc_model}")
print(f"   ROC-AUC Score: {best_roc_score:.4f}")

print("\n" + "="*80)
print("WHY THIS MODEL PERFORMS BEST")
print("="*80)

if best_auprc_model == best_roc_model:
    print(f"\n{best_auprc_model} is the overall best performer based on both AUPRC and ROC-AUC.")
else:
    print(f"\n{best_auprc_model} leads in AUPRC, while {best_roc_model} leads in ROC-AUC.")

print("\nKey Insights:")
print("\n1. AUPRC (Area Under Precision-Recall Curve) is MORE IMPORTANT for fraud detection")
print("   because the dataset is highly imbalanced (fraud = <1% of transactions).")
print("   AUPRC focuses on the minority class (fraud) performance.")

print("\n2. ROC-AUC measures overall discrimination ability but can be misleading")
print("   for imbalanced datasets. A model can have high ROC-AUC but still")
print("   perform poorly on the minority class.")

print("\n3. Model Rankings by AUPRC:")
auprc_ranking = comparison_df.sort_values('AUPRC', ascending=False)[['Model', 'AUPRC']]
for idx, row in auprc_ranking.iterrows():
    print(f"   {idx+1}. {row['Model']:25s} - AUPRC: {row['AUPRC']:.4f}")

print("\n4. Precision vs Recall Trade-off:")
for idx, row in comparison_df.iterrows():
    print(f"   {row['Model']:25s} - Precision: {row['Precision']:.4f}, Recall: {row['Recall']:.4f}")

print("\n5. Best Model Recommendation:")
print(f"   For fraud detection, prioritize {best_auprc_model} based on AUPRC.")
print("   This model achieves the best balance of precision and recall for")
print("   detecting fraudulent transactions in this highly imbalanced dataset.")

print("\n" + "="*80)

### Conclusion

This comparison demonstrates that different models have varying strengths for fraud detection:

- **Logistic Regression**: Simple, interpretable baseline with decent performance
- **Random Forest**: Strong ensemble method with good precision-recall balance
- **XGBoost**: Advanced gradient boosting with excellent fraud detection capability
- **ANN**: Deep learning approach with potential for complex pattern recognition

For production deployment in fraud detection systems, the model with the highest AUPRC should be prioritized, as it best handles the class imbalance challenge inherent in fraud detection problems.